In [ ]:
# ===== モード制御 =====
# "train" : アダプタ学習のみ
# "infer" : PEFT→vLLM変換 → test.csv 推論 → submission.zip
MODE = "infer"

# ===== 学習ハイパーパラメータ（MODE="train" 時のみ使用） =====
LORA_RANK     = 32      # competition 上限（vLLM max_lora_rank=32）
LORA_DROPOUT  = 0.0
EPOCHS        = 1
LR            = 2e-4
BATCH_SIZE    = 1
GRAD_ACCUM    = 32      # effective batch size = 32
MAX_SEQ       = 8192
SAVE_STEPS    = None
LOGGING_STEPS = 10
SUBSAMPLE     = None    # デバッグ時は 100 など

In [ ]:
# ===== 環境セットアップ + パス定義 =====
import os, subprocess, glob, sys, shutil

def _is_kaggle():
    return os.path.exists("/kaggle/input")

def _is_runpod():
    return os.environ.get("RUNPOD_POD_ID") is not None

ENV = "kaggle" if _is_kaggle() else "runpod" if _is_runpod() else "local"
print(f"[env] {ENV}  mode={MODE}")

if ENV == "runpod":
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    import json as _json
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as _f:
        _json.dump({"username": os.environ["KAGGLE_USERNAME"],
                    "key": os.environ["KAGGLE_KEY"]}, _f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

def _find_path(*candidates):
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return candidates[0]

def _find_adapter_path():
    """既学習アダプタのパスを検索（infer モード用）"""
    for pattern in [
        "/kaggle/input/models/shotokishida/nemotron-adapter/*/*/*",
        "/kaggle/input/*nemotron-adapter*",
    ]:
        hits = glob.glob(pattern)
        if hits:
            return hits[0]
    hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError("adapter が見つかりません。model_sources を確認してください。")

if ENV == "kaggle":
    NEMOTRON_MODEL_DIR   = _find_path(
        "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
        "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/Transformers/default/1")
    METRIC_UTIL_DIR      = "/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script"
    COMPETITION_DATA_DIR = _find_path(
        "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge",
        "/kaggle/input/nvidia-nemotron-model-reasoning-challenge",
        "/kaggle/input/nvidia-nemotron-3-reasoning-challenge")
    OUTPUT_DIR = "/kaggle/working"
    if MODE == "train":
        SCRIPT_SRC  = _find_path(
            "/kaggle/input/datasets/shotokishida/nemotron-train-scripts/train.py",
            "/kaggle/input/nemotron-train-scripts/train.py")
        SCRIPT_DST  = "/kaggle/working/train.py"
        COT_CSV     = _find_path(
            "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection/train_split_with_cot.csv",
            "/kaggle/input/nemotron-sft-lora-cot-selection/train_split_with_cot.csv")
        ADAPTER_OUT = "/kaggle/working/adapter"
        os.makedirs(ADAPTER_OUT, exist_ok=True)
        shutil.copy2(SCRIPT_SRC, SCRIPT_DST)
        print(f"SCRIPT     = {SCRIPT_DST}  exists={os.path.exists(SCRIPT_DST)}")
        print(f"COT_CSV    = {COT_CSV}  exists={os.path.exists(str(COT_CSV))}")
        print(f"ADAPTER_OUT= {ADAPTER_OUT}")
    else:
        ADAPTER_PATH = _find_adapter_path()
        print(f"ADAPTER_PATH={ADAPTER_PATH}  exists={os.path.exists(ADAPTER_PATH)}")
    print(f"NEMOTRON_MODEL_DIR={NEMOTRON_MODEL_DIR}  exists={os.path.exists(str(NEMOTRON_MODEL_DIR))}")
    print(f"COMPETITION_DATA_DIR={COMPETITION_DATA_DIR}  exists={os.path.exists(COMPETITION_DATA_DIR)}")
elif ENV == "runpod":
    NEMOTRON_MODEL_DIR   = "/workspace/models/nemotron-3-nano-30b-a3b-bf16"
    METRIC_UTIL_DIR      = "/workspace/data/nvidia-metric-utility-script"
    COMPETITION_DATA_DIR = "/workspace/data/nvidia-nemotron-competition"
    OUTPUT_DIR           = "/workspace/output"
    SCRIPT_DST  = "/workspace/repo/notebooks/nemotron-train/train.py"
    COT_CSV     = "/workspace/data/konbu17-cot/train_split_with_cot.csv"
    ADAPTER_OUT = "/workspace/output/adapter"
    ADAPTER_PATH= "/workspace/models/nemotron-adapter"
else:
    NEMOTRON_MODEL_DIR = None; METRIC_UTIL_DIR = None
    COMPETITION_DATA_DIR = "data/nvidia-nemotron-competition"
    OUTPUT_DIR = "output"
    SCRIPT_DST  = "notebooks/nemotron-train/train.py"
    COT_CSV     = "data/konbu17-cot/train_split_with_cot.csv"
    ADAPTER_OUT = "output/adapter"
    ADAPTER_PATH= "models/nemotron-adapter"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ===== [train] ライブラリインストール + 学習実行 =====
if MODE == "train":
    import glob as _g

    def _pip(*args):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

    try:
        import trl; print(f"trl: {trl.__version__}")
    except ImportError:
        _wheels = (_g.glob("/kaggle/input/datasets/shotokishida/nemotron-train-scripts/trl-*.whl") or
                   _g.glob("/kaggle/input/nemotron-train-scripts/trl-*.whl"))
        if _wheels:
            print(f"Installing trl from wheel: {_wheels[0]}")
            _pip("--no-deps", _wheels[0])
        else:
            _pip("trl")
        import trl; print(f"trl installed: {trl.__version__}")

    import pandas as pd
    df_cot = pd.read_csv(COT_CSV)
    print(f"CoT data: {df_cot.shape}  columns={df_cot.columns.tolist()}")
    print(df_cot["type"].value_counts().to_string())

    subsample_flag = f"--subsample {SUBSAMPLE}" if SUBSAMPLE else ""
    save_flag      = f"--save_steps {SAVE_STEPS}" if SAVE_STEPS else ""

    ret = subprocess.run(
        f'python -u "{SCRIPT_DST}"'
        f' --model_dir   "{NEMOTRON_MODEL_DIR}"'
        f' --data_csv    "{COT_CSV}"'
        f' --output_dir  "{ADAPTER_OUT}"'
        f' --lora_rank   {LORA_RANK}'
        f' --lora_dropout {LORA_DROPOUT}'
        f' --epochs      {EPOCHS}'
        f' --lr          {LR}'
        f' --batch_size  {BATCH_SIZE}'
        f' --grad_accum  {GRAD_ACCUM}'
        f' --max_seq_len {MAX_SEQ}'
        f' --logging_steps {LOGGING_STEPS}'
        f' --zip_output'
        f' {save_flag} {subsample_flag}',
        shell=True
    )
    if ret.returncode != 0:
        raise RuntimeError("学習スクリプトがエラーで終了しました")
    print(f"\n[train] 完了。アダプタ: {os.listdir(ADAPTER_OUT)}")
    print("次のステップ: shotokishida/nemotron-adapter にアダプタをアップロードしてください。")

In [ ]:
# ===== [infer] アダプタ変換（PEFT → vLLM） =====
# ① キーリネーム: base_model.model.model.* → base_model.model.backbone.*
# ② MoE expert unfuse: experts.w1/w2 → experts.N.up_proj/down_proj
if MODE == "infer":
    import re, json
    from safetensors import safe_open
    from safetensors.torch import save_file

    def convert_adapter(src_dir, dst_dir):
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy2(os.path.join(src_dir, "adapter_config.json"),
                     os.path.join(dst_dir, "adapter_config.json"))
        with open(os.path.join(dst_dir, "adapter_config.json")) as f:
            cfg = json.load(f)
        print(f"[adapter] target_modules: {cfg.get('target_modules')}")
        print(f"[adapter] lora_rank: {cfg.get('r')}")

        src_st = os.path.join(src_dir, "adapter_model.safetensors")
        raw = {}
        with safe_open(src_st, framework="pt", device="cpu") as f:
            for key in f.keys():
                raw[key] = f.get_tensor(key)
        print(f"[adapter] loaded {len(raw)} tensors")

        base_names = {re.sub(r"\.lora_[AB]\.weight$", "", k) for k in raw}
        tensors = {}
        for base in sorted(base_names):
            lora_A = raw[f"{base}.lora_A.weight"]
            lora_B = raw[f"{base}.lora_B.weight"]
            renamed = base.replace("base_model.model.model.", "base_model.model.backbone.")

            if ".experts.w1" in base or ".experts.w2" in base:
                if lora_A.shape[0] == 1:
                    lora_A = lora_A.expand(lora_B.shape[0], -1, -1).contiguous()
                elif lora_B.shape[0] == 1:
                    lora_B = lora_B.expand(lora_A.shape[0], -1, -1).contiguous()
                proj = "up_proj" if ".w1" in base else "down_proj"
                for i in range(lora_A.shape[0]):
                    exp = re.sub(r"\.experts\.w[12]", f".experts.{i}.{proj}", renamed)
                    tensors[f"{exp}.lora_A.weight"] = lora_A[i].contiguous()
                    tensors[f"{exp}.lora_B.weight"] = lora_B[i].contiguous()
                continue
            if lora_A.numel() == 0:
                continue
            tensors[f"{renamed}.lora_A.weight"] = lora_A
            tensors[f"{renamed}.lora_B.weight"] = lora_B

        dst_st = os.path.join(dst_dir, "adapter_model.safetensors")
        save_file(tensors, dst_st)
        print(f"[adapter] saved {len(tensors)} tensors → {dst_st}")

    convert_adapter(ADAPTER_PATH, OUTPUT_DIR)
    print("[adapter] conversion done")

In [ ]:
# ===== [infer] Triton / torch セットアップ（Blackwell GPU 対応） =====
if MODE == "infer":
    for cmd in [
        "uv pip uninstall torch torchvision torchaudio",
        f"tar -cf - -C {METRIC_UTIL_DIR} . | tar -xf - -C /tmp",
        "chmod +x /tmp/triton/backends/nvidia/bin/ptxas",
        "chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell",
    ]:
        print(f"Running: {cmd}")
        subprocess.run(cmd, shell=True, check=True)
    sys.path.insert(0, "/tmp")

In [ ]:
# ===== [infer] ヘルパー関数 + vLLM 初期化 =====
if MODE == "infer":
    import math, multiprocessing, time, re as _re
    import pandas as pd
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from pathlib import Path
    import kagglehub

    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"MODEL_PATH={MODEL_PATH}")

    def cache_model(path, exts=(".bin", ".pt", ".safetensors"), num_workers=16, chunk_mb=1024):
        def _warmup(fpath):
            total = 0
            with open(fpath, "rb") as f:
                while chunk := f.read(chunk_mb * 1024 * 1024):
                    total += len(chunk)
            return fpath, total
        path = Path(path)
        files = sorted(p for p in path.rglob("*") if p.is_file() and str(p).endswith(exts))
        print(f"[cache_model] {len(files)} file(s), {num_workers} worker(s)")
        t0, total_bytes = time.time(), 0
        with ThreadPoolExecutor(max_workers=num_workers) as pool:
            for i, fut in enumerate(as_completed({pool.submit(_warmup, f): f for f in files}), 1):
                fp, n = fut.result(); total_bytes += n
                print(f"[{i}/{len(files)}] cached {fp.name}")
        elapsed = time.time() - t0; gb = total_bytes / 1024**3
        print(f"[cache_model] total ≈ {gb:.2f} GB in {elapsed:.2f}s ({gb/elapsed:.2f} GB/s)")

    def extract_final_answer(text):
        if not text: return "NOT_FOUND"
        m = _re.findall(r"\\boxed\{([^}]*)(?:\}|$)", text)
        if m:
            non_empty = [x.strip() for x in m if x.strip()]
            return non_empty[-1] if non_empty else m[-1].strip()
        for pat in [r"The final answer is:\s*([^\n]+)", r"Final answer\s*[:：]\s*([^\n]+)"]:
            m = _re.findall(pat, text, _re.IGNORECASE)
            if m: return m[-1].strip()
        m = _re.findall(r"-?\d+(?:\.\d+)?", text)
        return m[-1] if m else text.strip().splitlines()[-1] if text.strip() else "NOT_FOUND"

    os.environ.update({
        "TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_FLAX": "1",
        "TRANSFORMERS_OFFLINE": "1", "CUDA_VISIBLE_DEVICES": "0",
        "TRITON_PTXAS_PATH": "/tmp/triton/backends/nvidia/bin/ptxas",
    })

    cache_model(MODEL_PATH)

    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest

    llm = LLM(
        model=str(MODEL_PATH), tensor_parallel_size=1,
        max_num_seqs=64, gpu_memory_utilization=0.85,
        dtype="auto", max_model_len=8192, trust_remote_code=True,
        enable_lora=True, max_lora_rank=32,
        enable_prefix_caching=True, enable_chunked_prefill=True,
    )
    sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=7680)
    print("[vllm] initialized")

In [ ]:
# ===== [infer] 推論実行（test.csv 全件） =====
if MODE == "infer":
    lora_path = OUTPUT_DIR
    print(f"lora_path={lora_path}  adapter_config exists={os.path.exists(os.path.join(lora_path, 'adapter_config.json'))}")

    tokenizer = llm.get_tokenizer()
    df = pd.read_csv(f"{COMPETITION_DATA_DIR}/test.csv")
    print(f"[inference] {len(df)} problems from test.csv")

    prompts = []
    for problem_text in df["prompt"]:
        content = problem_text + "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{{your answer}}`"
        try:
            prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": content}],
                tokenize=False, add_generation_prompt=True, enable_thinking=True,
            )
        except TypeError:
            prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": content}],
                tokenize=False, add_generation_prompt=True,
            )
        prompts.append(prompt)

    outputs = llm.generate(prompts, sampling_params, lora_request=LoRARequest("nemotron_adapter", 1, lora_path))
    df["answer"] = [extract_final_answer(out.outputs[0].text) for out in outputs]
    print(f"[inference] done. sample predictions:")
    print(df[["id", "answer"]].head(10).to_string(index=False))

In [ ]:
# ===== submission.zip 作成 =====
import zipfile as _zf

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")

if MODE == "train":
    # 学習モード: アダプタをzip（Kaggle model へのアップロード用）
    required = ["adapter_config.json", "adapter_model.safetensors"]
    with _zf.ZipFile(zip_path, "w", _zf.ZIP_DEFLATED) as zf:
        for fname in required:
            fpath = os.path.join(ADAPTER_OUT, fname)
            if not os.path.exists(fpath):
                raise FileNotFoundError(f"CRITICAL: {fname} が見つかりません")
            zf.write(fpath, fname)
    print(f"[train] submission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB  contents={required}")
    print("→ Kaggle model shotokishida/nemotron-adapter にアップロードしてください。")

else:  # infer
    # 推論モード: 変換済みアダプタをzip（コンペ提出用）
    required = ["adapter_config.json", "adapter_model.safetensors"]
    with _zf.ZipFile(zip_path, "w", _zf.ZIP_DEFLATED) as zf:
        for fname in required:
            fpath = os.path.join(OUTPUT_DIR, fname)
            if not os.path.exists(fpath):
                raise FileNotFoundError(f"CRITICAL: {fname} が見つかりません")
            zf.write(fpath, fname)
    print(f"[infer] submission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB  contents={required}")